In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [23]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [3]:
loader=PyPDFLoader("../data/medical_report.pdf")
docs=loader.load()

In [5]:
len(docs)

9

In [6]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_docs=splitter.split_documents(docs)


In [7]:
len(splitted_docs)

26

In [11]:
embediings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embediings
)

agents=tools,llm,prompt

In [34]:
@tool
def retriver_tool(query:str):

    """
        This Tool can Help you to retrieve the relevent data of the PDF document and these Medical Documents have Details about medical reports.
    """
    print("Tools called:",query)
    docs=vector_store.similarity_search(query=query,k=4)###k=number of releted chunks retived
    context=""

    for doc in docs:
        context = doc.page_content+"\n\n"
    return context


In [35]:
llm=ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [36]:
system_prompt="""
    You are a helpful assistent that answers questions using retrieved context.Always use the `retriver_tool` tool for questions requiring external knowledge.
"""

In [37]:
agent=create_agent(
    model=llm,
    tools=[retriver_tool],
    system_prompt=system_prompt
)

In [40]:
query="What is the name of patient,and what is the name of Doctors"
res=agent.invoke({"messages":[{"role":"user","content":query}]})

Tools called: patient name doctor name
Tools called: Ms NIKITA CHUDHARY DR NITIN NAHAR doctor patient


In [41]:
print(res["messages"][-1].text)

Based on the medical report, here are the details:

* **Patient Name:** Ms. NIKITA CHUDHARY
* **Doctor(s) Mentioned:**
  * **Referring Doctor (Ref By):** Dr. Nitin Nahar
  * **Laboratory Doctors / Pathologists:** 
    * Dr. Swapnil Gupta (MD, Pathology - Chief of Laboratory)
    * Dr. Kiran Bhargava Pathak (MD, Pathology - Chief of Laboratory)
